# Athena Database

In [1]:
database_name = "ndbc_data_lake"
table_name = "stdmet_buoy_data"

%store -r CURATED_PREFIX
%store -r bucket

In [3]:
import boto3

athena_results_prefix = "athena-query-results/"

s3 = boto3.client("s3")
s3.put_object(Bucket=bucket, Key=(athena_results_prefix))
output_location = f"s3://{bucket}/{athena_results_prefix}"
print(f"Athena query results will go to: {output_location}")

Athena query results will go to: s3://sagemaker-us-east-1-318401170150/athena-query-results/


In [17]:
import time

athena = boto3.client("athena")
database_name = "ndbc_data"

ddl_database = f"CREATE DATABASE IF NOT EXISTS {database_name}"

response = athena.start_query_execution(
    QueryString=ddl_database,
    ResultConfiguration={"OutputLocation": output_location}
)

query_execution_id = response["QueryExecutionId"]

# Wait for query to finish
while True:
    status = athena.get_query_execution(QueryExecutionId=query_execution_id)
    state = status["QueryExecution"]["Status"]["State"]
    if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
        break
    time.sleep(2)

if state == "SUCCEEDED":
    print(f"Database {database_name} created successfully!")
else:
    print("Database creation failed:", status["QueryExecution"]["Status"]["StateChangeReason"])


Database ndbc_data created successfully!


In [40]:
table_name = "ndbc_stdmet"
ddl_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    visibility double,
    PTDY double,
    tide double,
    wind_speed_ms double,
    wave_energy double,
    buoy string
)
STORED AS PARQUET
LOCATION 's3://{bucket}/curated/ndbc/buoy=46086/';
"""

In [33]:
response = athena.start_query_execution(
    QueryString=ddl_query,
    ResultConfiguration={"OutputLocation": output_location}
)

query_execution_id = response["QueryExecutionId"]

# Wait for query to finish
while True:
    status = athena.get_query_execution(QueryExecutionId=query_execution_id)
    state = status["QueryExecution"]["Status"]["State"]
    if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
        break
    time.sleep(2)

if state == "SUCCEEDED":
    print(f"Athena table {database_name}.{table_name} created successfully!")
else:
    print("Query failed:", status["QueryExecution"]["Status"]["StateChangeReason"])


Athena table ndbc_data.ndbc_stdmet created successfully!


In [42]:
response = athena.start_query_execution(
    QueryString=ddl_query,
    ResultConfiguration={"OutputLocation": output_location}
)

query_execution_id = response["QueryExecutionId"]

# Wait for query to finish
while True:
    status = athena.get_query_execution(QueryExecutionId=query_execution_id)
    state = status["QueryExecution"]["Status"]["State"]
    if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
        break
    time.sleep(2)

if state == "SUCCEEDED":
    results = athena.get_query_results(QueryExecutionId=query_execution_id)
    for row in results["ResultSet"]["Rows"]:
        print([col.get("VarCharValue", "") for col in row["Data"]])
else:
    print("Query failed:", status["QueryExecution"]["Status"]["StateChangeReason"])

In [38]:
rows = results["ResultSet"]["Rows"]

import pandas as pd
# First row = column names
headers = [col.get("VarCharValue") for col in rows[0]["Data"]]

# Remaining rows = data
data = [
    [col.get("VarCharValue") for col in row["Data"]]
    for row in rows[1:]
]

print(pd.DataFrame(data, columns=headers))

Empty DataFrame
Columns: [record_id, event_time, station_id, wind_direction, wind_speed, wind_gust, wave_height, dominant_wave_period, average_wave_period, mean_wave_direction, pressure, air_temperature, water_temperature, dewpoint_temperature, visibility, tide, wind_speed_ms, wave_energy, buoy]
Index: []


In [36]:
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="curated/ndbc/"
)
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

curated/ndbc/buoy=46086/stdmet.parquet 534615


## Release Resources

In [1]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>

In [2]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>